In [ ]:
%cd ../..
import os
import polars as pl
import numpy as np
import random
import itertools
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, BatchSampler
from tqdm import tqdm

from evaluation import *

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
embeddings_path = "/scratch/scratch1/embeddings/RSNA-PE/demo"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
embed_dim = 768

def get_embedding(series_uid, slice_idx):
    x = torch.load(os.path.join(embeddings_path, f"{series_uid}.pth"), mmap=True)
    return x["cls"][slice_idx]

def get_embedding_stack(series_uid):
    x = torch.load(os.path.join(embeddings_path, f"{series_uid}.pth"), mmap=True)
    return x["cls"]

class RSNAPE(Dataset):
    def __init__(self, labels_df):
        self.labels_df = labels_df

    def __len__(self):
        return len(self.labels_df)
    
    def get_labels(self):
        return self.labels_df["has_pe"].to_list()
    
    def __getitem__(self, idx):
        series_uid, slice_idx, has_pe, _ = self.labels_df.row(idx)
        embedding = get_embedding(series_uid, slice_idx)
        label = torch.tensor(has_pe, dtype=torch.float32)
        return embedding, label
    
class RSNAPEWholeVolume(Dataset):
    def __init__(self, labels_df, p=1.0, sigma=0.1):
        self.labels_df = labels_df
        self.p = p
        self.sigma = sigma

    def __len__(self):
        return len(self.labels_df)
    
    def get_labels(self):
        return self.labels_df["has_pe"].to_list()
    
    def add_embedding_noise(self, embeddings):
        if torch.rand(1).item() < self.p:
            noise = torch.randn_like(embeddings) * self.sigma
            embeddings = embeddings + noise
        return embeddings
    
    def __getitem__(self, idx):
        series_uid, has_pe = self.labels_df.row(idx)
        embedding = get_embedding_stack(series_uid)
        embedding = self.add_embedding_noise(embedding)
        return embedding, has_pe
    
class StratifiedBatchSampler(BatchSampler):
    """A custom sampler to ensure each batch has a 50/50 class distribution."""
    def __init__(self, labels, batch_size, drop_last=False):
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.drop_last = drop_last

        assert batch_size % 2 == 0, "Batch size must be even for equal stratification."
        self.half_bs = self.batch_size // 2

        self.pos_indices = np.where(self.labels)[0].tolist()
        self.neg_indices = np.where(~self.labels)[0].tolist()
        
        self.min_class_len = min(len(self.pos_indices), len(self.neg_indices))
        self.num_batches = self.min_class_len // self.half_bs

    def __iter__(self):
        pos = np.random.permutation(self.pos_indices).tolist()[:self.min_class_len]
        neg = np.random.permutation(self.neg_indices).tolist()[:self.min_class_len]

        for i in range(0, self.min_class_len, self.half_bs):
            if i + self.half_bs > self.min_class_len and self.drop_last:
                break
            batch = pos[i:i+self.half_bs] + neg[i:i+self.half_bs]
            np.random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches

In [ ]:
class PoolerMlp(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.pooler = MultiHeadAttentionPool(embed_dim, num_heads)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(embed_dim, 1)

    def forward(self, x, mask=None):
        x = self.pooler(x, mask)
        x = self.dropout(x)
        return self.fc1(x)
    
class Mlp(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.mlp(x)

In [ ]:
print("Loading initial data (single slice)")
labels_df = pl.read_csv(os.path.join(embeddings_path, "labels.csv"))
labels_df = labels_df.filter((pl.col("num_img_pe") == 0) | (pl.col("num_img_pe") > 4))
print(f"Full dataset has {len(labels_df)} samples.")

In [ ]:
print("Loading initial data (whole volume)")
labels_df = pl.read_csv(os.path.join(embeddings_path, "labels.csv"))

all_series_uids = labels_df["series_uid"].unique().to_list()
pos_series_uids = labels_df.filter(pl.col("has_pe"))["series_uid"].unique().to_list()
neg_series_uids = [sid for sid in all_series_uids if sid not in pos_series_uids]

labels_dict = {"series_uid": pos_series_uids + neg_series_uids}
labels_dict.update({"has_pe": [True] * len(pos_series_uids) + [False] * len(neg_series_uids)})

labels_df = pl.DataFrame(labels_dict)
print(f"Full dataset has {len(labels_df)} samples.")

In [ ]:
train_size = 0.8
series_uids = labels_df["series_uid"].unique().to_list()
random.shuffle(series_uids)
num_train_samples = int(train_size * len(series_uids))
train_series_uids = series_uids[:num_train_samples]
val_series_uids = series_uids[num_train_samples:]
train_df_full = labels_df.filter(pl.col("series_uid").is_in(train_series_uids))
val_df_full = labels_df.filter(pl.col("series_uid").is_in(val_series_uids))

print(f"Training samples: {len(train_df_full)}, Validation samples: {len(val_df_full)}")

In [ ]:
train_dataset = RSNAPEWholeVolume(train_df_full)
val_dataset = RSNAPEWholeVolume(val_df_full)

neg_count = (np.array(train_dataset.get_labels())==0).sum()
pos_count = (np.array(train_dataset.get_labels())==1).sum()
pos_weight = torch.tensor(neg_count / (pos_count + 1e-8), dtype=torch.float32).to(device)

In [ ]:
param_grid = {
    'learning_rate': [1e-3],
    'num_heads': [16],
    'dropout': [0.5],
    'weight_decay': [1e-2],
    'batch_size': [512]
}

keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

grid_search_results = []
best_rocauc = -1
best_params = None
num_epochs = 20

print(f"Searching {len(param_combinations)} hyperparameter combinations...")
for i, params in enumerate(param_combinations):
    print(f"\nCombination {i+1}/{len(param_combinations)}: {params}")
    
    train_sampler = StratifiedBatchSampler(train_dataset.get_labels(), params['batch_size'], drop_last=True)
    train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=collate_classification_stack)
    val_dataloader = DataLoader(val_dataset, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_classification_stack)

    model = PoolerMlp(embed_dim, params["num_heads"], params["dropout"]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    output = train_classifier_stack(
        model, optimizer, loss_fn, train_dataloader, val_dataloader,
        num_epochs=num_epochs, device=device, select_criteria="rocauc"
    )
    
    max_val_rocauc = max(output['val_rocauc'])
    params['best_val_rocauc'] = max_val_rocauc
    grid_search_results.append(params)

    print(f"Best ROC AUC: {max_val_rocauc:.04f}")
    
    if max_val_rocauc > best_rocauc:
        best_rocauc = max_val_rocauc
        best_params = params

print("\n--- Grid Search Complete ---")
print(f"Best Validation ROC UAC: {best_rocauc:.4f}")
print(f"Best Hyperparameters: {best_params}")

results_df = pd.DataFrame(grid_search_results)
print("\nFull Grid Search Results:")
print(results_df.sort_values(by='best_val_rocauc', ascending=False).to_string())

In [ ]:
data_ratios = [0.2, 0.4, 0.6, 0.8, 1.0]
data_size_results = []
hidden_dim = 512
batch_size = 512
learning_rate = 1e-3
weight_decay = 1e-2
dropout = 0.5
num_epochs = 15

for ratio in data_ratios:
    print(f"\n--- Training with {ratio*100:.0f}% of the data ---")
    
    num_select_samples = int(ratio * len(series_uids))
    current_series_uids = series_uids[:num_select_samples]
    num_train_subset = int(train_size * len(current_series_uids))
    train_uids_subset = current_series_uids[:num_train_subset]
    train_df_subset = labels_df.filter(pl.col("series_uid").is_in(train_uids_subset))
    val_df_subset = val_df_full
    
    print(f"Training samples: {len(train_df_subset)}, Validation samples: {len(val_df_subset)}")

    train_dataset = RSNAPEWholeVolume(train_df_subset)
    val_dataset = RSNAPEWholeVolume(val_df_subset)
    train_sampler = StratifiedBatchSampler(train_dataset.get_labels(), batch_size, drop_last=True)
    train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=collate_classification_stack)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_classification_stack)

    model = PoolerMlp(embed_dim, hidden_dim, dropout).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    output = train_classifier_stack(
        model, optimizer, loss_fn, train_dataloader, val_dataloader,
        num_epochs=num_epochs, device=device, select_criteria="f1"
    )
    
    final_val_rocauc = max(output['val_rocauc'])
    data_size_results.append({'data_ratio': ratio, 'num_train_samples': len(train_df_subset), 'val_rocauc': final_val_rocauc})
    print(f"Validation ROC AUC for {ratio*100:.0f}% data: {final_val_rocauc:.4f}")
    
print("\n--- Data Size Testing Complete ---")
data_size_df = pd.DataFrame(data_size_results)
print(data_size_df.to_string(index=False))

plt.figure()
plt.plot(data_size_df['num_train_samples'], data_size_df['val_rocauc'], marker='o')
plt.title('Model Performance vs. Training Data Size')
plt.xlabel('Number of Training Samples')
plt.ylabel('Best Validation ROC AUC')
plt.grid(True)
plt.show()

In [ ]:
num_epochs = 30
embed_dim = 768
num_heads = 4
batch_size = 512
learning_rate = 1e-4
weight_decay = 1e-2
dropout = 0.5

train_dataset = RSNAPEWholeVolume(train_df_full)
val_dataset = RSNAPEWholeVolume(val_df_full)
train_sampler = StratifiedBatchSampler(train_dataset.get_labels(), batch_size, drop_last=True)
train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=collate_classification_stack)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_classification_stack)

model = PoolerMlp(embed_dim, num_heads, dropout).to(device)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

output = train_classifier_stack(
    model, optimizer, loss_fn, train_dataloader, val_dataloader,
    num_epochs=num_epochs, device=device, select_criteria="rocauc"
)


In [ ]:
plot_train_curves(output["train_rocauc"], output["val_rocauc"], "ROC AUC")

In [ ]:
plot_train_curves(output["train_f1"], output["val_f1"], "F1 Score")

In [ ]:
all_labels, all_predictions = get_predictions_stack(model, val_dataloader, device)

In [ ]:
all_predictions = all_predictions > 0.0

plot_confusion_matrix(all_labels, all_predictions, True)